# 🚀 GPT

In this notebook, we'll walk through the steps required to train your own GPT model on the recipe review dataset

The code is adapted from the excellent [GPT tutorial](https://keras.io/examples/generative/text_generation_with_miniature_gpt/) created by Apoorv Nandan available on the Keras website.  
https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/09_transformer/gpt/gpt.ipynb

https://www.kaggle.com/datasets/hugodarwood/epirecipes

The training runs an order of magnitude faster on colab (with TPU enabled) than on a laptop.

In [278]:
#%load_ext autoreload
#%autoreload 2
import numpy as np
import json
import re
import string
from IPython.display import display, HTML
#!pip install kaggle
import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks
from tensorflow import keras

import os

## 0. Parameters <a name="parameters"></a>

In [279]:
VOCAB_SIZE = 10000
MAX_LEN = 128          
EMBEDDING_DIM = 128   
KEY_DIM = 128          
N_HEADS = 4            
FEED_FORWARD_DIM = 512 
VALIDATION_SPLIT = 0.2 
SEED = 57
LOAD_MODEL = False
BATCH_SIZE = 32        
EPOCHS = 100 

## 1. Load the data <a name="load"></a>

In [280]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hugodarwood/epirecipes")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Simon Seo\.cache\kagglehub\datasets\hugodarwood\epirecipes\versions\2


In [281]:
# Find JSON files in the directory
json_files = [f for f in os.listdir(path) if f.endswith('.json')]

if json_files:
    # Load the first JSON file (or specify which one you want)
    json_file_path = os.path.join(path, json_files[0])
    print(f"\nLoading: {json_files[0]}")
    
    with open(json_file_path, 'r', encoding='utf-8') as json_data:
        recipe = json.load(json_data)
    
    print(f"Successfully loaded data with {len(recipe)} entries")
else:
    print("\nNo JSON files found. The dataset might be in another format (CSV, etc.)")


Loading: full_format_recipes.json
Successfully loaded data with 20130 entries


In [282]:
recipe[10]

{'directions': ['Heat oil in heavy large skillet over medium-high heat. Add shallots and minced rosemary and sauté until tender, about 3 minutes. Add yams and broth to skillet and bring to boil. Cover skillet, reduce heat to medium-low and simmer until yams are almost tender, about 10 minutes. Add cream and sprinkle lightly with nutmeg. Simmer uncovered until yams are very tender and liquid thickens and coats yams, about 4 minutes. Season with salt and pepper. (Can be made 1 day ahead. Transfer to microwave-safe dish. Chill until cold, then cover and keep chilled. Rewarm, covered, in microwave on medium-low heat.)'],
 'fat': 5.0,
 'date': '2004-08-20T04:00:00.000Z',
 'categories': ['Milk/Cream',
  'Dairy',
  'Side',
  'Thanksgiving',
  'Rosemary',
  'Sweet Potato/Yam',
  'Fall',
  'Nutmeg',
  'Simmer',
  'Bon Appétit'],
 'calories': 256.0,
 'desc': 'Simmering the yams fills them with flavor and yields a lovely coating.',
 'protein': 4.0,
 'rating': 3.75,
 'title': 'Yams Braised with Cr

In [283]:
filtered_data = [
    f"Recipe: {x['title']} | Category: {x['categories'][0]} | Rating: {x['rating']} | Description: {x['desc']} | Main Ingredient: {x['ingredients'][0]}"
    for x in recipe
    if isinstance(x, dict)
    and x.get('categories') and len(x['categories']) > 0
    and x.get('title')
    and x.get('ingredients') and len(x['ingredients']) > 0
    and x.get('desc')
]

In [284]:
# Count the recipes
n_recipes = len(filtered_data)
print(f"{n_recipes} recipes loaded")

13427 recipes loaded


In [285]:
example = filtered_data[25]
print(example)

Recipe: Deviled Ham  | Category: Food Processor | Rating: 3.125 | Description: Can be prepared in 45 minutes or less. | Main Ingredient: 3/4 pound cooked ham, cut into pieces


## 2. Tokenize the data <a name="tokenize"></a>

In [286]:
# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}, '\n'])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s

text_data = [pad_punctuation(x) for x in filtered_data]

In [287]:
# Display an example of a recipe
example_data = text_data[25]
example_data

'Recipe : Deviled Ham | Category : Food Processor | Rating : 3 . 125 | Description : Can be prepared in 45 minutes or less . | Main Ingredient : 3 / 4 pound cooked ham , cut into pieces'

In [288]:
# Convert to a Tensorflow Dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)

In [289]:
# Create a vectorisation layer
vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)

In [290]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [291]:
# Display some token:word mappings
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

0: 
1: [UNK]
2: :
3: |
4: .
5: ,
6: and
7: the
8: recipe
9: -


In [292]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

[   8    2 2433  361    3   12    2  153  318    3   14    2   23    4
  105    3   13    2   44   42   92   22  101   82   32   93    4    3
   10   11    2   23   21   19   81  197  361    5   80   61  194    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0]


## 3. Create the Training Set <a name="create"></a>

In [293]:
# Create the training set of recipes and the same text shifted by one word
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y

train_ds = text_ds.map(prepare_inputs)

In [294]:
example_input_output = train_ds.take(1).get_single_element()

In [295]:
# Example Input
example_input_output[0][0]

<tf.Tensor: shape=(128,), dtype=int64, numpy=
array([  8,   2, 589, 795, 150,  37, 392,   3,  12,   2, 220,   3,  14,
         2,  19,   4,  27,   3,  13,   2, 342, 115,   2, 569, 267, 231,
        20, 275,   2, 101, 267,   3,  10,  11,   2, 160,  88, 233,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0])>

In [296]:
# Example Output (shifted by one token)
example_input_output[1][0]

<tf.Tensor: shape=(128,), dtype=int64, numpy=
array([  2, 589, 795, 150,  37, 392,   3,  12,   2, 220,   3,  14,   2,
        19,   4,  27,   3,  13,   2, 342, 115,   2, 569, 267, 231,  20,
       275,   2, 101, 267,   3,  10,  11,   2, 160,  88, 233,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0])>

## 5. Create the causal attention mask function <a name="causal"></a>

In [297]:
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    m = i >= j - n_src + n_dest
    mask = tf.cast(m, dtype)
    mask = tf.reshape(mask, [1, n_dest, n_src])
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0
    )
    return tf.tile(mask, mult)

np.transpose(causal_attention_mask(1, 10, 10, dtype=tf.int32)[0])

array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]], dtype=int32)

## 6. Create a Transformer Block layer <a name="transformer"></a>

In [298]:
class TransformerBlock(layers.Layer):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1):
        super(TransformerBlock, self).__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate
        self.attn = layers.MultiHeadAttention(
            num_heads, key_dim, output_shape=embed_dim
        )
        self.dropout_1 = layers.Dropout(self.dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn_1 = layers.Dense(self.ff_dim, activation="relu")
        self.ffn_2 = layers.Dense(self.embed_dim)
        self.dropout_2 = layers.Dropout(self.dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        causal_mask = causal_attention_mask(
            batch_size, seq_len, seq_len, tf.bool
        )
        attention_output, attention_scores = self.attn(
            inputs,
            inputs,
            attention_mask=causal_mask,
            return_attention_scores=True,
        )
        attention_output = self.dropout_1(attention_output)
        out1 = self.ln_1(inputs + attention_output)
        ffn_1 = self.ffn_1(out1)
        ffn_2 = self.ffn_2(ffn_1)
        ffn_output = self.dropout_2(ffn_2)
        return (self.ln_2(out1 + ffn_output), attention_scores)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "key_dim": self.key_dim,
                "embed_dim": self.embed_dim,
                "num_heads": self.num_heads,
                "ff_dim": self.ff_dim,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config

## 7. Create the Token and Position Embedding <a name="embedder"></a>

In [299]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "max_len": self.max_len,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

## 8. Build the Transformer model <a name="transformer_decoder"></a>

In [300]:
inputs = layers.Input(shape=(None,), dtype=tf.int32)
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x, attention_scores = TransformerBlock(
    N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM
)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
gpt = models.Model(inputs=inputs, outputs=[outputs, attention_scores])
gpt.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss=[losses.SparseCategoricalCrossentropy(), None]
)

In [301]:
gpt.summary()

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_9 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_9  │ (None, None, 128)      │     1,296,384 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_9             │ [(None, None, 128),    │       396,032 │
│ (TransformerBlock)              │ (None, 4, None, None)] │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, None, 10000)    │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,982,416 (11.38 MB)

 Trainable params: 2,982,416 (11.38 MB)

 Non-trainable params: 0 (0.00 B)

In [302]:
if LOAD_MODEL:
    # model.load_weights('./models/model')
    gpt = models.load_model("./models/gpt", compile=True)

## 9. Train the Transformer <a name="train"></a>

In [ ]:
# Create a TextGenerator checkpoint
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }

    def sample_from(self, probs, temperature):
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y, att = self.model.predict(x, verbose=0)
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append(
                {
                    "prompt": start_prompt,
                    "word_probs": probs,
                    "atts": att[0, :, -1, :],
                }
            )
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("Recipe : title", max_tokens=50, temperature=.7)

In [304]:
# Create a model save checkpoint
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.weights.h5", # ckpt",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

# Tokenize starting prompt
text_generator = TextGenerator(vocab)

In [305]:
gpt.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)

Epoch 1/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - loss: 6.6802
generated text:
Recipe : title , . | main 

420/420 ━━━━━━━━━━━━━━━━━━━━ 53s 123ms/step - loss: 5.1310
Epoch 2/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - loss: 2.7934
generated text:
Recipe : title with ancestors | category : 2 with fish | rating : 4 . 4 : 2 tablespoons | description : 2 / 4 . | main ingredient : 1 / 4 cup onion 

420/420 ━━━━━━━━━━━━━━━━━━━━ 52s 124ms/step - loss: 2.5979
Epoch 3/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 2.2333
generated text:
Recipe : title - salad with finely delight | category : bon appétit | rating : 3 . 375 | description : a little an khao of a rich , and a salad . | main ingredient : 1 large loves the potatoes ( about 1 / 2 or less

420/420 ━━━━━━━━━━━━━━━━━━━━ 54s 128ms/step - loss: 2.1875
Epoch 4/100
420/420 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - loss: 2.0622
generated text:
Recipe : title cream blessings if with special . | category : rice | rating : 0 | des

In [306]:
# Save the final model
#gpt.save("./models/gpt")
gpt.save("./gpt.keras")

# 3. Generate text using the Transformer

In [307]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        highlighted_text = []
        for word, att_score in zip(
            i["prompt"].split(), np.mean(i["atts"], axis=0)
        ):
            highlighted_text.append(
                '<span style="background-color:rgba(135,206,250,'
                + str(att_score / max(np.mean(i["atts"], axis=0)))
                + ');">'
                + word
                + "</span>"
            )
        highlighted_text = " ".join(highlighted_text)
        display(HTML(highlighted_text))

        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

In [308]:
max_tok = MAX_LEN
info = text_generator.generate(
    "Recipe: Rating: 3.5", max_tokens=max_tok, temperature=1.0
)


generated text:
Recipe: Rating: 3.5 and cognac house rolls | category . | rating : 5 . 0 | description : figs are all over the time favorites : this quick , simply mix of crisp , chile powder and crisp , hard that are available at supermarkets . | main ingredient : 1 / 2 cup olive oil 



In [309]:
j=0.2
while(j<=1):
    for i in range(2):
        print(f"Temperature: {j}")
        info = text_generator.generate(    
            "Recipe: title:", max_tokens=max_tok, temperature=j
        )
    j += 0.2

Temperature: 0.2

generated text:
Recipe: title: johnson . | category : bon appétit | rating : 0 . 0 | description : in philippine cane sugar , german , sweet , and homey , and homey meals , and delicious . we ' ve been obsessed with wild mushrooms and wild mushrooms growing up in the country , where they are shocked when i like to use it [UNK] sweeter than [UNK] dosa - like experience . | main ingredient : 1 1 1 / 2 pounds lean ground pork 

Temperature: 0.2

generated text:
Recipe: title: johnson . | category : coffee | rating : 0 . 0 | description : in trinidad , this is a resealable plastic wrap in a prime rib , but we tested this recipe with a . | main ingredient : 2 tablespoons ( 1 / 4 stick ) unsalted butter 

Temperature: 0.4

generated text:
Recipe: title: johnson . | category : coffee | rating : 0 . 0 | description : in trinidad , this is a jammy punch recipe with orange - in philadelphia - fruit sorbets and is from [UNK] . | main ingredient : 2 cups sugar 

Temperature: 0.4


In [310]:
j = 0.2
while(j<=6):
    for i in range(3):
        print(f"Temperature: {j}")
        info = text_generator.generate(    
            "Recipe: title:", max_tokens=max_tok, temperature=j
        )
    j += 0.1

Temperature: 0.2

generated text:
Recipe: title: johnson . | category : beer | rating : 0 . 0 | description : in trinidad , this pie dough is a resealable plastic wrap in a first thanksgiving pie dough , in a pie crust . | main ingredient : 2 ( 3 / 4 - to 3 - inch - thick ) pieces 

Temperature: 0.2

generated text:
Recipe: title: johnson . | category : bon appétit | rating : 0 . 0 | description : as much as in drinking vinegars , we ' re asked julius than their [UNK] , with their [UNK] , and [UNK] . | main ingredient : 1 / 2 cup sugar 

Temperature: 0.2

generated text:
Recipe: title: johnson . | category : beer | rating : 0 . 0 | description : we love the entire menu , there ' s no - like corn and stuff it : it super - easy and easy - to - make . | main ingredient : 1 / 2 cup mayonnaise 

Temperature: 0.30000000000000004

generated text:
Recipe: title: johnson . | category : coffee | rating : 0 . 0 | description : this drink was featured as a cocktail of the month . click here to lea

In [311]:
print_probs(info, vocab)

johnson:   	0.09000000357627869%
.:   	0.07000000029802322%
florida:   	0.07000000029802322%
world:   	0.07000000029802322%
hotel:   	0.07000000029802322%
--------



cocktail:   	0.14000000059604645%
back:   	0.10000000149011612%
':   	0.09000000357627869%
[UNK]:   	0.09000000357627869%
salad:   	0.09000000357627869%
--------



salad:   	0.18000000715255737%
-:   	0.1599999964237213%
|:   	0.15000000596046448%
of:   	0.15000000596046448%
[UNK]:   	0.12999999523162842%
--------



-:   	0.12999999523162842%
|:   	0.11999999731779099%
cocktail:   	0.11999999731779099%
mayonnaise:   	0.10000000149011612%
dressing:   	0.09000000357627869%
--------



|:   	0.1599999964237213%
chocolate:   	0.07999999821186066%
soup:   	0.07999999821186066%
chicken:   	0.07999999821186066%
vinaigrette:   	0.07999999821186066%
--------



...

|:   	0.4300000071525574%
":   	0.15000000596046448%
for:   	0.14000000059604645%
dressing:   	0.14000000059604645%
.:   	0.14000000059604645%
--------



|:   	0.3799999952316284%
it:   	0.27000001072883606%
.:   	0.20999999344348907%
you:   	0.20000000298023224%
the:   	0.18000000715255737%
--------



|:   	2.369999885559082%
.:   	0.6200000047683716%
with:   	0.41999998688697815%
and:   	0.3499999940395355%
for:   	0.3499999940395355%
--------



mayonnaise:   	0.10000000149011612%
iced:   	0.09000000357627869%
soaking:   	0.09000000357627869%
blender:   	0.09000000357627869%
coleslaw:   	0.07999999821186066%
--------



|:   	1.3799999952316284%
with:   	0.2800000011920929%
by:   	0.1899999976158142%
for:   	0.18000000715255737%
<:   	0.18000000715255737%
--------



I ran 100 epochs. I think I overtrained it. But, yeah I played with different parameters, but the runs take too long so, I'll stop here.